# OpenSTEF Domain Baselines — XGB and LGB

Use Python 3.11, otherwise may have package issue

## 1. Setup

In [13]:

# !pip install openstef

In [14]:
from pathlib import Path
import os
import sys
import warnings
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
import tqdm
from openstef.data_classes.model_specifications import ModelSpecificationDataClass
from openstef.data_classes.prediction_job import PredictionJobDataClass
from openstef.pipeline.train_model import train_model_pipeline
from openstef.pipeline.create_forecast import create_forecast_pipeline

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "requirements").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "load"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from src.metrics import print_metrics

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "Load-Train.xlsx"
TEST_PATH = DATA_DIR / "Load-Test.xlsx"
MLFLOW_DIR = ROOT / "models" / "openstef"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_URI = str(MLFLOW_DIR / "mlflow_trained_models")
ARTIFACT_DIR = str(MLFLOW_DIR / "mlflow_artifacts")

QUANTILES = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
MODEL_CONFIGS = {
    "xgb": {"id": 287, "output": OUTPUT_DIR / "openstef_xgb.csv"},
    "lgb": {"id": 285, "output": OUTPUT_DIR / "openstef_lgb.csv"},
}

os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.seterr(over="ignore")

## 2. Data preparation

In [15]:
def prepare_openstef_frame(df, base_year=2019):
    out = df.copy()
    out["timestamp"] = pd.to_datetime(
        {
            "year": base_year + out["Year"],
            "month": out["Month"],
            "day": out["Day"],
            "hour": out["Hour"] - 1,
        },
        errors="raise",
    )
    out = out.rename(columns={"Load": "load"})
    out["temp"] = out[[
        "Site-1 Temp", "Site-2 Temp", "Site-3 Temp", "Site-4 Temp", "Site-5 Temp"
    ]].mean(axis=1)
    out["radiation"] = out[[
        "Site-1 GHI", "Site-2 GHI", "Site-3 GHI", "Site-4 GHI", "Site-5 GHI"
    ]].mean(axis=1)
    return out[["load", "temp", "radiation", "timestamp"]]


def make_daily_future_windows(df):
    windows = []
    tmp = df.copy()
    tmp["timestamp"] = pd.to_datetime(tmp["timestamp"])
    for date, g in tmp.groupby(tmp["timestamp"].dt.floor("D")):
        if len(g) == 24:
            g = g.set_index("timestamp").sort_index()
            g.index = g.index.tz_localize("UTC")
            windows.append((date, g))
    return windows

context_raw = pd.read_excel(TRAIN_PATH)
test_raw = pd.read_excel(TEST_PATH)

context_flat = prepare_openstef_frame(context_raw)
test_flat = prepare_openstef_frame(test_raw)

context_df = context_flat.set_index("timestamp").sort_index()
context_df.index = context_df.index.tz_localize("UTC")
context_df = context_df.asfreq("h")

test_df = test_flat.set_index("timestamp").sort_index()
test_df.index = test_df.index.tz_localize("UTC")

test_daily = make_daily_future_windows(test_flat)
print("Training rows:", len(context_df))
print("Test rows:", len(test_df))
print("Complete test days:", len(test_daily))

## 3. OpenSTEF rolling backtest

In [11]:
def make_prediction_job(model_name, job_id):
    # Exact prediction-job settings from the source notebook.
    pj = dict(
        id=job_id,
        model=model_name,
        quantiles=QUANTILES,
        forecast_type="demand",
        lat=32.7,
        lon=117.2,
        horizon_minutes=24 * 60,
        resolution_minutes=60,
        name="load_forecast",
        save_train_forecasts=True,
    )
    return PredictionJobDataClass(**pj)


def run_openstef_backtest(model_name, job_id):
    pj = make_prediction_job(model_name, job_id)
    _ = ModelSpecificationDataClass(id=pj["id"])

    # Train exactly once on the original context set.
    train_model_pipeline(
        pj,
        context_df,
        check_old_model_age=False,
        mlflow_tracking_uri=MLFLOW_TRACKING_URI,
        artifact_folder=ARTIFACT_DIR,
    )

    rolling_results = []
    context = context_df.copy()

    with open(os.devnull, "w") as fnull, redirect_stdout(fnull):
        for _, test_day in tqdm.tqdm(test_daily):
            context = pd.concat([context, test_day])
            context.loc[test_day.index, "load"] = np.nan

            forecast = create_forecast_pipeline(
                pj,
                context,
                mlflow_tracking_uri=MLFLOW_TRACKING_URI,
            )
            rolling_results.append(forecast)

            # Reveal realized load before the next daily forecast.
            context.loc[test_day.index, "load"] = test_day["load"]

    return (
        pd.concat(rolling_results)
        .reset_index()
        .rename(columns={"index": "timestamp"})
    )

## 4. Standardize outputs and evaluate

In [7]:
QUANTILE_MAP = {
    "quantile_P10": 0.1,
    "quantile_P20": 0.2,
    "quantile_P30": 0.3,
    "quantile_P40": 0.4,
    "quantile_P50": 0.5,
    "quantile_P60": 0.6,
    "quantile_P70": 0.7,
    "quantile_P80": 0.8,
    "quantile_P90": 0.9,
}


def standardize_openstef_output(forecast, truth):
    out = forecast.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True)

    truth_df = truth.reset_index()[["timestamp", "load"]].copy()
    truth_df["timestamp"] = pd.to_datetime(truth_df["timestamp"], utc=True)
    out = out.merge(truth_df, on="timestamp", how="left")
    out = out.rename(columns={"load": "target"})

    for col, q in QUANTILE_MAP.items():
        out[str(q)] = out[col]
    out["prediction"] = out["forecast"]

    keep = ["timestamp", "target", "prediction"] + [str(q) for q in QUANTILES]
    return out[keep].sort_values("timestamp").reset_index(drop=True)

## 5. Run XGB and LGB

In [16]:
results = {}

for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\nRunning OpenSTEF {model_name.upper()} ...")
    raw = run_openstef_backtest(model_name, cfg["id"])
    result = standardize_openstef_output(raw, test_df)

    print_metrics(result, quantiles=QUANTILES, label=f"OpenSTEF {model_name.upper()}")
    result.to_csv(cfg["output"], index=False)
    print("Saved:", cfg["output"])

    results[model_name] = result